# 승부예측용 데이터셋 생성

## Imports

In [1]:
import pandas as pd
import json
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import os

# matplotlib 한글 폰트 설정
if os.name == 'nt':
    plt.rc('font', family='Malgun Gothic')
elif os.name == 'posix':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)

## 챔피언 태그 확인

In [2]:
# read chamnpion_tag
champ_tag = pd.read_csv('champ_tag.csv', index_col=0)
champ_tag.drop(columns=['name'], inplace=True)
champ_tag.fillna(0, inplace=True)
champ_tag = champ_tag.astype(int)

In [3]:
tag_lineup_dict = {}
for tag_name in champ_tag.columns.to_list():
    tag_lineup_dict['me_'+tag_name] = []
    tag_lineup_dict['en_'+tag_name] = []
tag_lineup_dict['win'] = []

## 데이터 추출 및 적재

In [4]:
# read match data

match_list = pd.read_csv('../data/match_id_list.csv')['match_id'].to_list()

for match_id in tqdm(match_list):
    with open(f'../data/match/{match_id}.json', 'r') as f:
        match_data = json.load(f)
    # 다시하기 제외
    if match_data['info']['gameDuration'] < 300:
        continue
    
    # BLUE 팀기준
    for tag_name in champ_tag.columns.to_list():
        tag_lineup_dict['me_'+tag_name].append(0)
        tag_lineup_dict['en_'+tag_name].append(0)
    for participant in match_data['info']['participants']:
        if participant['teamId'] == 100:
            team = 'me'
        else:
            team = 'en'
        
        for tag_name in champ_tag.columns.to_list():
            if participant['championId'] in champ_tag[champ_tag[tag_name] == 1].index:
                tag_lineup_dict[f'{team}_{tag_name}'][-1] += 1
    tag_lineup_dict['win'].append(match_data['info']['participants'][0]['win'])

    # RED 팀기준

    for tag_name in champ_tag.columns.to_list():
        tag_lineup_dict['me_'+tag_name].append(0)
        tag_lineup_dict['en_'+tag_name].append(0)
    for participant in match_data['info']['participants']:
        if participant['teamId'] == 200:
            team = 'me'
        else:
            team = 'en'
        
        for tag_name in champ_tag.columns.to_list():
            if participant['championId'] in champ_tag[champ_tag[tag_name] == 1].index:
                tag_lineup_dict[f'{team}_{tag_name}'][-1] += 1
    tag_lineup_dict['win'].append(not match_data['info']['participants'][0]['win'])

# dataframe으로 변환
tag_lineup_df = pd.DataFrame(tag_lineup_dict)
# 확인
tag_lineup_df.head()

  0%|          | 0/60209 [00:00<?, ?it/s]

,me_전사,en_전사,me_암살자,en_암살자,me_마법사,en_마법사,me_원거리 공격,en_원거리 공격,me_지원가,en_지원가,...,en_글로벌,me_이동기,en_이동기,me_은신,en_은신,me_버프,en_버프,me_그랩,en_그랩,win
0,1,2,1,1,3,3,2,1,1,2,...,0,4,3,0,0,1,1,0,1,True
1,2,1,1,1,3,3,1,2,2,1,...,1,3,4,0,0,1,1,1,0,False
2,2,2,2,0,2,2,2,0,2,3,...,0,3,2,1,1,1,2,0,1,False
3,2,2,0,2,2,2,0,2,3,2,...,1,2,3,1,1,2,1,1,0,True
4,2,2,1,2,2,1,2,2,2,1,...,0,4,3,0,1,0,1,1,0,True


In [5]:
tag_lineup_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119656 entries, 0 to 119655
Data columns (total 43 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   me_전사      119656 non-null  int64
 1   en_전사      119656 non-null  int64
 2   me_암살자     119656 non-null  int64
 3   en_암살자     119656 non-null  int64
 4   me_마법사     119656 non-null  int64
 5   en_마법사     119656 non-null  int64
 6   me_원거리 공격  119656 non-null  int64
 7   en_원거리 공격  119656 non-null  int64
 8   me_지원가     119656 non-null  int64
 9   en_지원가     119656 non-null  int64
 10  me_탱커      119656 non-null  int64
 11  en_탱커      119656 non-null  int64
 12  me_AP      119656 non-null  int64
 13  en_AP      119656 non-null  int64
 14  me_AD      119656 non-null  int64
 15  en_AD      119656 non-null  int64
 16  me_CC      119656 non-null  int64
 17  en_CC      119656 non-null  int64
 18  me_유틸      119656 non-null  int64
 19  en_유틸      119656 non-null  int64
 20  me_돌진      119656 non-null

## 저장

In [6]:
tag_lineup_df.to_csv('WinPredDataset.csv')